# Stage 02 — Data Quality Assessment

## Enterprise Data Quality & Governance Project

### Objective
The purpose of this stage is to formally assess the quality of the Customer 360 dataset using defined Data Quality rules.

The assessment covers four dimensions:
- Completeness
- Uniqueness
- Validity
- Consistency

**Accuracy** and **Timeliness** are not formally assessed because the dataset does not provide an external source of truth or business metadata such as an expected refresh frequency.


## 1. Load and Prepare the Dataset


In [ ]:
import pandas as pd

df = pd.read_csv("customers.csv")
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

df.head()


## 2. Define Data Quality Rules

A total of 15 rules were defined across the four selected Data Quality dimensions.


In [ ]:
rules_catalog = [
    {"rule_id": "DQ001", "dimension": "Completeness", "field": "customer_id", "rule": "Must not contain NULL"},
    {"rule_id": "DQ002", "dimension": "Completeness", "field": "age", "rule": "Must not contain NULL"},
    {"rule_id": "DQ003", "dimension": "Completeness", "field": "gender", "rule": "Must not contain NULL"},
    {"rule_id": "DQ004", "dimension": "Completeness", "field": "location", "rule": "Must not contain NULL"},
    {"rule_id": "DQ005", "dimension": "Completeness", "field": "income_level", "rule": "Must not contain NULL"},
    {"rule_id": "DQ006", "dimension": "Completeness", "field": "signup_date", "rule": "Must not contain NULL"},
    {"rule_id": "DQ007", "dimension": "Uniqueness", "field": "customer_id", "rule": "Must be unique"},
    {"rule_id": "DQ008", "dimension": "Validity", "field": "age", "rule": "Must be between 18 and 69"},
    {"rule_id": "DQ009", "dimension": "Validity", "field": "gender", "rule": "Must be Male or Female"},
    {"rule_id": "DQ010", "dimension": "Validity", "field": "income_level", "rule": "Must be High, Medium, or Low"},
    {"rule_id": "DQ011", "dimension": "Validity", "field": "location", "rule": "Must belong to approved locations"},
    {"rule_id": "DQ012", "dimension": "Validity", "field": "signup_date", "rule": "Must contain valid dates"},
    {"rule_id": "DQ013", "dimension": "Consistency", "field": "income_level", "rule": "Must use consistent capitalization"},
    {"rule_id": "DQ014", "dimension": "Consistency", "field": "location", "rule": "Must use standardized location names"},
    {"rule_id": "DQ015", "dimension": "Consistency", "field": "signup_date", "rule": "Must use a consistent date format"}
]

rules_catalog_df = pd.DataFrame(rules_catalog)
rules_catalog_df


## 3. Define Approved Values


In [ ]:
allowed_gender = ["Male", "Female"]
allowed_income = ["High", "Medium", "Low"]
allowed_locations = ["Mumbai", "Bangalore", "Chennai", "Hyderabad", "Kolkata", "Delhi"]


## 4. Create the Quality Results Structure

Each rule produces a standardized result containing total records, violations, valid records, quality rate, and status.


In [ ]:
quality_results = []
total_records = len(df)


## 5. Completeness Checks


In [ ]:
completeness_rules = [
    ("DQ001", "customer_id"),
    ("DQ002", "age"),
    ("DQ003", "gender"),
    ("DQ004", "location"),
    ("DQ005", "income_level"),
    ("DQ006", "signup_date")
]

for rule_id, field in completeness_rules:
    violations = df[field].isnull().sum()
    valid_records = total_records - violations
    quality_rate = (valid_records / total_records) * 100

    quality_results.append({
        "rule_id": rule_id,
        "dimension": "Completeness",
        "field": field,
        "total_records": total_records,
        "violations": violations,
        "valid_records": valid_records,
        "quality_rate": quality_rate,
        "status": "PASS" if violations == 0 else "FAIL"
    })


## 6. Uniqueness Check


In [ ]:
violations = df["customer_id"].duplicated().sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100

quality_results.append({
    "rule_id": "DQ007",
    "dimension": "Uniqueness",
    "field": "customer_id",
    "total_records": total_records,
    "violations": violations,
    "valid_records": valid_records,
    "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})


## 7. Validity Checks


In [ ]:
# DQ008 — Age range
violations = ((df["age"] < 18) | (df["age"] > 69)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ008", "dimension": "Validity", "field": "age",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ009 — Gender
violations = (~df["gender"].isin(allowed_gender)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ009", "dimension": "Validity", "field": "gender",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ010 — Income
violations = (~df["income_level"].isin(allowed_income)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ010", "dimension": "Validity", "field": "income_level",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ011 — Location
violations = (~df["location"].isin(allowed_locations)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ011", "dimension": "Validity", "field": "location",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ012 — Valid dates
violations = df["signup_date"].isnull().sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ012", "dimension": "Validity", "field": "signup_date",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})


## 8. Consistency Checks


In [ ]:
# DQ013 — Income capitalization consistency
violations = (~df["income_level"].isin(allowed_income)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ013", "dimension": "Consistency", "field": "income_level",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ014 — Standardized location names
violations = (~df["location"].isin(allowed_locations)).sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ014", "dimension": "Consistency", "field": "location",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})

# DQ015 — Date consistency after parsing
violations = df["signup_date"].isnull().sum()
valid_records = total_records - violations
quality_rate = (valid_records / total_records) * 100
quality_results.append({
    "rule_id": "DQ015", "dimension": "Consistency", "field": "signup_date",
    "total_records": total_records, "violations": violations,
    "valid_records": valid_records, "quality_rate": quality_rate,
    "status": "PASS" if violations == 0 else "FAIL"
})


## 9. Quality Results


In [ ]:
quality_results = pd.DataFrame(quality_results)
quality_results


## 10. Overall Assessment Summary


In [ ]:
total_rules = len(quality_results)
total_violations = quality_results["violations"].sum()
overall_quality_rate = ((total_records * total_rules - total_violations) / (total_records * total_rules)) * 100
overall_status = "PASS" if total_violations == 0 else "FAIL"

summary = {
    "total_records": total_records,
    "total_rules": total_rules,
    "total_violations": total_violations,
    "overall_quality_rate": overall_quality_rate,
    "overall_status": overall_status
}

summary


### Final Result

The final assessment produced:
- **15 Data Quality rules**
- **0 violations**
- **100% overall quality rate**
- **PASS** status

The earlier profiling observations (`LOW` and `Chennal`) were treated as standardization/consistency observations and were resolved before the final validated state documented in the project.


## 11. Assessment Limitations

### Accuracy
Accuracy could not be formally assessed because there was no external trusted source or reference dataset against which customer attributes could be compared.

### Timeliness
Timeliness could not be formally assessed because the dataset does not contain a `last_updated_date` field or a defined business refresh expectation.

These limitations are documented rather than treated as data quality failures.


# Stage 02 Conclusion

The dataset was formally assessed using 15 Data Quality rules across Completeness, Uniqueness, Validity, and Consistency.

The final validated state achieved a **100% quality rate with 0 violations across all 15 rules**. This demonstrates that a successful Data Quality assessment does not require finding a large number of errors; clearly defined rules, measurable results, and documented limitations are what make the assessment meaningful.

The results from this stage provide the foundation for **Stage 03 — Data Quality Issues & Findings**.
